# Lab 4: Reasoning with Probability

Everything so far has been about searching: a space of boards, of tours, of moves. This lab is about what to do when you are not looking for an answer but weighing up how likely one is.

There are three short exercises, and then you use them to take apart a famous piece of statistical evidence. The purpose is to practice basic probabilities in a data science format, in prep for probabilistic AI algorithms. We will also use pandas, so if you need a review let me know.

**What to submit:** commit this notebook with its outputs. Try to finish as much as possible by Friday.

**How the exercises work.** Each one asks you to compute some numbers. There is no function you have to write and no check cell, so feel free to use a format that's comfortable. There are hints and **the last hint gives the answers**, so you can confirm you got there. Try before you open it.

## Part 0: Where a probability comes from

Two files sit next to this notebook. The first is the passenger list of the Titanic — one row per person, the same file used in about a thousand introductory data courses.

In [ ]:
# PROVIDED
import matplotlib.pyplot as plt
import pandas as pd

titanic = pd.read_csv("titanic.csv")
print(len(titanic), "passengers", len(titanic.columns), "columns")
titanic.head()

The data contains several variables, but we are just interested in the following three:

* `Survived`: 1 or 0
* `Pclass`: 1, 2, or 3. Denotes the class of the passenger
* `Sex`: `"male"` or `"female"`

(Note that some of the variables have missing data, like Cabin or age.)

In [ ]:
# PROVIDED
people = titanic[["Survived", "Pclass", "Sex"]]
print('missing data:')
print(people.isna().sum().to_string())
print()
print(people.head())

### Using counts to calculate probabilities

`P(survived)` is the proportion of rows with `Survived == 1`. Because that column is only ever 0 or 1, we can simply use the mean to compute this (think about why).

In [ ]:
# PROVIDED
print(f"rows where Survived == 1 : {(people.Survived == 1).sum()}")
print(f"rows in total            : {len(people)}")
print(f"P(survived)              : {people.Survived.mean():.4f}")

### A joint distribution is a contingency table

A **joint distribution** says how likely every *combination* of values is. We can do this conveniently using the groupby function of the dataframe. This counts the number of times each combination of sex, Pclass, and survived occured. Then we divide by the total number of counts to get the joint probabilities.

In [ ]:
# PROVIDED
counts = people.groupby(["Sex", "Pclass", "Survived"]).size()
joint = counts / counts.sum()
print('counts:')
print(counts)
print()
print('joint:')
print(joint.to_string())
print(f"\nsum = {joint.sum():.6f}")

That table is the whole probabilistic description of these 891 people. Every question in Parts 1 to 3 is answered by working with this table.

## Part 1: Marginalizing

To **marginalize** is to drop a variable by adding up over all of its values.

For example, in the table above, `("female", 1, 1)`,  `("female", 2, 1)` and `("female", 3, 1)` describe women who survived in class 1, class 2, and class 3, respectively. If we want to find the probability a woman in *any* class survived, we need to add them up.

### Exercise 1

Compute two things:

1. the joint distribution over **sex and survival only**. This should be a 2x2 "table" whose entries sum to 1.
2. `P(survived)` and `P(female)` on their own, each obtained by marginalizing further

In [ ]:
# Your code here.

<details><summary>Hint 1: grouping by fewer things</summary>

The cell in Part 0 grouped by all three variables. Group by only the two you want to keep, count, and divide by the total.

</details>

<details><summary>Hint 2: marginalizing again</summary>

Once you have the four-row table, `P(survived)` is the sum of the two rows where `Survived` is 1. On a grouped Series, `.groupby("Survived").sum()` will do that for you. Or, you can go back to the original data and take the mean of the column.

</details>

<details><summary>Hint 3: the answers</summary>

The joint over sex and survival:

| Sex | Survived | probability |
|---|---|---|
| female | 0 | 0.0909 |
| female | 1 | 0.2615 |
| male | 0 | 0.5253 |
| male | 1 | 0.1223 |

and marginalizing those: `P(survived) = 0.3838`, `P(female) = 0.3524`.

</details>

## Part 2: Conditioning

To **condition** is to throw away every row that contradicts what you have been told, and then treat what is left as the whole world.

If you learn a passenger was female, the 577 rows describing men become impossible so must be excluded from the probability calculation. `P(survived | female)` is the share of the *remaining* 314 rows that survived, rather than the share of all 891.

When conditioning, be careful about what you divide by!

### Exercise 2

Compute:

1. `P(survived | female)` and `P(survived | male)`
2. `P(survived | Pclass = 1)`, `P(survived | Pclass = 2)` and `P(survived | Pclass = 3)`

In [ ]:
# Your code here.

<details><summary>Hint 1: keeping only some rows</summary>

`people[people.Sex == "female"]` gives you back just the rows for women. Everything else follows from that smaller table.

</details>

<details><summary>Hint 2: the second step</summary>

Having kept only the women, the answer is the share of *those* rows that survived, so take the mean of `Survived` on the smaller table, not the original one. If you find yourself dividing by 891, you have conditioned on nothing.

</details>

<details><summary>Hint 3: the answers</summary>

`P(survived | female) = 0.7420` from 314 women, `P(survived | male) = 0.1889` from 577 men.

By class: `0.6296` from 216 in first, `0.4728` from 184 in second, `0.2424` from 491 in third.

</details>

Based on your results, what would you conclude about the probability of men surviving vs that of women?

*your answer here*

## Part 3: Conditioning on two things at once

Nothing changes when the evidence gets longer. Keep the rows that satisfy *all* of it, then take the proportion.

### Exercise 3

Compute `P(survived | female, Pclass)` and `P(survived | male, Pclass)` for each of the three classes (2x3 numbers).

Then answer, in the markdown cell below: being female is worth a great deal, and travelling first class is worth a great deal. Is being a first-class woman worth the two added together?

In [ ]:
# Your code here.

*Your answer here.*

<details><summary>Hint 1: combining conditions</summary>

`people[(people.Sex == "female") & (people.Pclass == 1)]` keeps the rows that satisfy both. The brackets around each comparison are needed.

</details>

<details><summary>Hint 2: the answers</summary>

| | class 1 | class 2 | class 3 |
|---|---|---|---|
| female | 0.9681 (n=94) | 0.9211 (n=76) | 0.5000 (n=144) |
| male | 0.3689 (n=122) | 0.1574 (n=108) | 0.1354 (n=347) |


</details>

Based on this new result, would you change your answer from Part 2?

*your answer here*

## Part 4: A test that is 99% accurate

Put the Titanic aside for a moment.

A disease affects 1 person in 1,000. There is a test for it that is **99% accurate in both directions**: if you have the disease it comes back positive 99 times out of 100, and if you do not it comes back negative 99 times out of 100.

You take the test. It comes back positive.

**Before running the next cell, write down what you think the chance is that you have the disease.** (This is just a guess.)

*your answer here*

In [ ]:
# PROVIDED
# 100,000 people, counted out.
people_tested = 100_000
ill = people_tested // 1000                       # 1 in 1,000
healthy = people_tested - ill

screening = pd.DataFrame([
    {"disease": True,  "test": "positive", "count": round(ill * 0.99)},
    {"disease": True,  "test": "negative", "count": round(ill * 0.01)},
    {"disease": False, "test": "positive", "count": round(healthy * 0.01)},
    {"disease": False, "test": "negative", "count": round(healthy * 0.99)},
])
print(screening.to_string(index=False))

positives = screening[screening.test == "positive"]

**Calculate P( test positive | disease ) and P(test negative | not disease)**. This is the accuracy of the test. Is it 99%?

In [ ]:
# YOUR CODE

**Calculate P( disease | test positive )**. If I test positive, what is the probability I have the disease? Does it match your prediction?

## Part 5: Berkeley, 1973

The second file is real admissions data. In 1973 the graduate school at the University of California, Berkeley was accused of discriminating against women applicants. These are the six largest departments, which between them handled about a third of all applicants.

In [ ]:
# PROVIDED
berkeley = pd.read_csv("berkeley.csv")
print(berkeley.to_string(index=False))
print(f"\n{berkeley.Freq.sum()} applicants")

Each row is a count of people, so we need to keep track of `Freq` in our calculations.

### Exercise 4

Compute the share of men admitted and the share of women admitted, across all six departments together.

In [ ]:
# Your code here.

<details><summary>Hint 1: summing counts</summary>

Group by gender and admission status and sum `Freq`. That gives you four numbers: admitted and rejected, for each gender.

</details>

<details><summary>Hint 2: the answers</summary>

`P(admitted | Male) = 0.4452`, from 1,198 admitted of 2,691 applicants.
`P(admitted | Female) = 0.3035`, from 557 admitted of 1,835 applicants.

</details>

Are male or females more likely to get admitted? It seems men are more likely to be admitted from the numbers, but let's see what happens if we look at one department at at time.

In [ ]:
# PROVIDED
table = berkeley.pivot_table(index=["Dept", "Gender"], columns="Admit",
                             values="Freq", aggfunc="sum")
rates = (table["Admitted"] / table.sum(axis=1)).unstack()

print(f"{'dept':>5} {'men':>8} {'women':>8}   higher")
for dept in rates.index:
    men, women = rates.loc[dept, "Male"], rates.loc[dept, "Female"]
    print(f"{dept:>5} {men:>8.1%} {women:>8.1%}   {'women' if women > men else 'men'}")

The direction reverses. Pooled together the men do better; taken department by department, the women do better in four of the six.

This is **Simpson's paradox**: an association that holds within every group can weaken, vanish or flip when the groups are pooled. It is not a rounding artefact or a trick. Both statements are literally true of the same 4,526 people.

The next cell shows where it comes from.

In [ ]:
# PROVIDED
dept_totals = berkeley.groupby("Dept").Freq.sum()
dept_admitted = berkeley[berkeley.Admit == "Admitted"].groupby("Dept").Freq.sum()
dept_women = berkeley[berkeley.Gender == "Female"].groupby("Dept").Freq.sum()

admit_rate = dept_admitted / dept_totals
women_share = dept_women / dept_totals

print(f"{'dept':>5} {'admit rate':>12} {'share of applicants who were women':>36}")
for dept in admit_rate.index:
    print(f"{dept:>5} {admit_rate[dept]:>12.1%} {women_share[dept]:>35.1%}")

In [ ]:
# PROVIDED
figure, axes = plt.subplots(1, 2, figsize=(11, 4))

positions = range(len(rates.index))
axes[0].bar([p - 0.2 for p in positions], rates["Male"], width=0.4,
            label="men")
axes[0].bar([p + 0.2 for p in positions], rates["Female"], width=0.4,
            label="women")
axes[0].set_xticks(list(positions))
axes[0].set_xticklabels(rates.index)
axes[0].set_xlabel("department")
axes[0].set_ylabel("share admitted")
axes[0].legend()

axes[1].scatter(women_share, admit_rate, s=90, color="#009E73")
for dept in admit_rate.index:
    axes[1].annotate(dept, (women_share[dept], admit_rate[dept]),
                     xytext=(7, 4), textcoords="offset points")
axes[1].set_xlabel("share of applicants who were women")
axes[1].set_ylabel("share of applicants admitted")

figure.tight_layout()

The first figure above shows that women tend to do better in 4 out of 6 departments. The second figure shows that there is a negative correlation between the admission rate of the department and the share of applicants who are women. That is, women tended to apply to more difficult departments.